# Deploy Models from Model Registry to Endpoints

In the previous lab, we registered two versions of our bikeshare model and assigned aliases (champion/challenger).

In this notebook we:
1. Retrieve the champion model from the registry by alias
2. Create an endpoint
3. Deploy the champion model to the endpoint
4. Test with an online prediction
5. Deploy the challenger alongside with a traffic split
6. Shift 100% traffic to the challenger
7. Undeploy the old champion

In [ ]:
from google.cloud import aiplatform

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"      # Replace with your GCP project ID
REGION = "us-central1"

aiplatform.init(project=PROJECT_ID, location=REGION)

## 1. Retrieve the Champion Model by Alias

In [ ]:
MODEL_DISPLAY_NAME = "bikeshare-model"

models = aiplatform.Model.list(filter=f'display_name="{MODEL_DISPLAY_NAME}"')
model_resource_name = models[0].resource_name
print(f"Found model: {model_resource_name}")

registry = aiplatform.models.ModelRegistry(model=model_resource_name)

champion_model = registry.get_model(version="champion")
print(f"Champion - Version: {champion_model.version_id}")
print(f"Resource: {champion_model.resource_name}")

## 2. Create an Endpoint

In [ ]:
endpoint = aiplatform.Endpoint.create(
    display_name="bikeshare-prediction-endpoint",
)

print(f"Endpoint created: {endpoint.display_name}")
print(f"Resource name: {endpoint.resource_name}")

## 3. Deploy the Champion to the Endpoint

In [ ]:
endpoint.deploy(
    model=champion_model,
    deployed_model_display_name="champion-deployment",
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=100,
)

print("Champion deployed with 100% traffic")

## 4. Test with an Online Prediction

In [ ]:
instance = [0.24, 0.81] + [0] * 43 + [1] + [0] * 4  # 50 features

prediction = endpoint.predict(instances=[instance])
print(f"Predicted log-count: {prediction.predictions}")

## 5. Deploy the Challenger Alongside with a Traffic Split
Route 90% of traffic to the champion and 10% to the challenger for A/B testing.

In [ ]:
challenger_model = registry.get_model(version="challenger")
print(f"Challenger - Version: {challenger_model.version_id}")

In [ ]:
endpoint.deploy(
    model=challenger_model,
    deployed_model_display_name="challenger-deployment",
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=10,
)

print("Challenger deployed with 10% traffic, champion now has 90%")

In [ ]:
traffic = endpoint.traffic_split
print(f"Current traffic split: {traffic}")

## 6. Promote Challenger: Shift 100% Traffic
The challenger passed testing. Route all traffic to it.

In [ ]:
deployed_models = endpoint.list_models()

champion_deployed_id = None
challenger_deployed_id = None

for dm in deployed_models:
    if dm.display_name == "champion-deployment":
        champion_deployed_id = dm.id
    elif dm.display_name == "challenger-deployment":
        challenger_deployed_id = dm.id

print(f"Champion deployed model ID: {champion_deployed_id}")
print(f"Challenger deployed model ID: {challenger_deployed_id}")

In [ ]:
endpoint.update(traffic_split={
    challenger_deployed_id: 100,
})

print(f"Updated traffic split: {endpoint.traffic_split}")
print("Challenger now serving 100% of traffic")

## 7. Undeploy the Old Champion
With 0% traffic, the old champion is just costing money. Remove it.

In [ ]:
endpoint.undeploy(deployed_model_id=champion_deployed_id)

print("Old champion undeployed")
print(f"Remaining deployed models: {[dm.display_name for dm in endpoint.list_models()]}")